# Análise exploratória do dataset

Métricas, mosaico e distribuições das 40 imagens — insumo para a seção 'Base de dados' do relatório.


In [1]:
import sys
from pathlib import Path

# O kernel inicia na pasta DESTE caderno — localiza a raiz do projeto:
RAIZ = Path.cwd()
if not (RAIZ / "src" / "leitura.py").exists():
    RAIZ = RAIZ.parent          # kernel começou dentro de notebooks/
sys.path.insert(0, str(RAIZ))   # raiz no path -> 'from src...' funciona

# Imports padrão de todos os cadernos:
from src.leitura import (PASTA_IMAGENS, PASTA_RESULTADOS, carregar_imagem,
                         nome_da_mascara, criar_pastas_resultados, auditar_base)
import cv2
import numpy as np
import matplotlib.pyplot as plt

print("raiz do projeto:", RAIZ)

raiz do projeto: /home/isadora/Documentos/ProcessIMg


In [ ]:
import csv

SAIDA = PASTA_RESULTADOS / "exploratoria"
SAIDA.mkdir(exist_ok=True)

def medir(caminho):
    """Métricas de uma imagem: brilho, área da lesão e contraste em 7 canais."""
    img = carregar_imagem(caminho)
    msk = cv2.imread(str(nome_da_mascara(caminho)), cv2.IMREAD_GRAYSCALE)
    cinza = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    r, g, b = img[:, :, 0], img[:, :, 1], img[:, :, 2]
    h, s, v = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]

    dentro, fora = msk > 0, msk == 0   # máscara booleana: lesão × pele
    contr = {
        "cinza": abs(cinza[fora].mean() - cinza[dentro].mean()),
        "R": abs(r[fora].mean() - r[dentro].mean()),
        "G": abs(g[fora].mean() - g[dentro].mean()),
        "B": abs(b[fora].mean() - b[dentro].mean()),
        "H": abs(h[fora].mean() - h[dentro].mean()),
        "S": abs(s[fora].mean() - s[dentro].mean()),
        "V": abs(v[fora].mean() - v[dentro].mean()),
    }
    return {
        "nome": caminho.name,
        "base": "ISIC" if caminho.name.startswith("ISIC") else "PH2",
        "largura": img.shape[1], "altura": img.shape[0],
        "megapixels": round(img.shape[0] * img.shape[1] / 1e6, 2),
        "cinza_media": round(float(cinza.mean()), 1),
        "cinza_dp": round(float(cinza.std()), 1),
        "area_lesao_frac": round(float(dentro.mean()), 4),
        **{f"contraste_{k}": round(val, 1) for k, val in contr.items()},
        "canal_vencedor": max(contr, key=contr.get),
    }

linhas = [medir(c) for c in sorted(PASTA_IMAGENS.iterdir())]
with open(SAIDA / "exploratoria_metricas.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(linhas[0].keys()))
    w.writeheader()
    w.writerows(linhas)
print(f"{len(linhas)} imagens medidas — CSV em {SAIDA / 'exploratoria_metricas.csv'}")

In [ ]:
ordem = sorted(PASTA_IMAGENS.iterdir())
fig, axes = plt.subplots(8, 5, figsize=(15, 24))
for ax, caminho in zip(axes.ravel(), ordem):
    img = carregar_imagem(caminho)
    thumb = cv2.resize(img, (120, 90), interpolation=cv2.INTER_AREA)
    ax.imshow(thumb)
    base = "ISIC" if caminho.name.startswith("ISIC") else "PH2"
    ax.set_title(caminho.stem, fontsize=7,
                 color="tab:blue" if base == "ISIC" else "tab:orange")
    ax.axis("off")
fig.suptitle("Base curada — 40 pares (azul: ISIC, laranja: PH2)", fontsize=14)
fig.savefig(SAIDA / "exploratoria_mosaico.png", bbox_inches="tight", dpi=90)
plt.show()
plt.close(fig)

In [ ]:
areas = [l["area_lesao_frac"] for l in linhas]
contr = [l["contraste_cinza"] for l in linhas]
medias = [l["cinza_media"] for l in linhas]
vitorias = {}
for l in linhas:
    vitorias[l["canal_vencedor"]] = vitorias.get(l["canal_vencedor"], 0) + 1

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes[0, 0].hist(areas, bins=16, color="tab:purple", edgecolor="white")
axes[0, 0].set_title("Fração de área da lesão (máscara de referência)")
axes[1, 0].hist(contr, bins=16, color="tab:green", edgecolor="white")
axes[1, 0].set_title("Contraste lesão×pele no cinza (|Δ média|)")
axes[0, 1].hist(medias, bins=16, color="tab:red", edgecolor="white")
axes[0, 1].set_title("Brilho médio da imagem (cinza)")
chaves = ["R", "G", "B", "H", "S", "V", "cinza"]
valores = [vitorias.get(k, 0) for k in chaves]
axes[1, 1].bar(chaves, valores, color="tab:blue", edgecolor="white")
axes[1, 1].set_title("Canal vencedor em contraste (nº de imagens, de 40)")
for i, v in enumerate(valores):
    if v:
        axes[1, 1].text(i, v + 0.3, str(v), ha="center")
for ax in axes.ravel():
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(SAIDA / "exploratoria_distribuicoes.png", bbox_inches="tight", dpi=110)
plt.show()
plt.close(fig)

### Conclusões da análise exploratória (medidas nas 40 imagens)

**Composição:** 30 ISIC + 10 PH2; resolução de 0,39 a 29,45 MP (variação de ~75× —
redimensionamento padrão no pré-processamento é obrigatório para um k de mediana fixo).

**Área da lesão:** 1,1%–80,3% (mediana 15,9%) — distribuição assimétrica: maioria das
lesões é pequena (<20% da imagem), com cauda de lesões grandes.

**Contraste lesão×pele (cinza):** 12,7–130,3 (mediana 46,6). **12 imagens com contraste
≤ 30** formam o grupo de dificuldade real — incluindo ISIC_0012940 (12,7, a pior da base).

**Canal vencedor nas 40 imagens: S = 23, B = 11, H = 4, G = 1, R = 1.** S e B juntos
explicam 85% das imagens; **o cinza padrão e o V nunca vencem**. Conclusão metodológica:
a segmentação deve ser avaliada prioritariamente nos canais **S** e **B**, não no cinza
(BT.601 é calibrado para o olho humano, não para lesão de pele).

**Artefatos visíveis no mosaico:** pelos (≥ 7 imagens — justifica a mediana do Passo 9),
marcações de caneta (ISIC_0013198), adesivo/marcador ciano (ISIC_0006982), bolhas de gel,
réguas (7 imagens) e vinheta circular nas PH2. Esses artefatos são regiões escuras que
podem ser confundidas com lesão pelo Otsu — o pós-processamento (maior componente conexo)
será essencial.

**Hipótese registrada para a Fase 3:** Otsu no canal S (reserva: B) deve superar o Otsu
no cinza; o grupo de contraste ≤ 30 deve concentrar as falhas.